# Optuna Hyperparameter Optimization for ResNet18 on CIFAR-10

> В этом ноутбуке использована Optuna для оптимизации гиперпараметров модели ResNet18 на наборе данных CIFAR10. Подбор таких гиперпараметров, как скорость обучения, размер пакета, весовое затухание и другие.

## Импорт необходимых библиотек

In [2]:
import torch
import numpy as np
import random
from typing import Tuple, List
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import CIFAR10
import torch.nn as nn
from torchvision.models.resnet import ResNet18_Weights
import torchvision.models as models
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
import wandb
import os
import optuna
import json

## Добавление обработки CUDA и воспроизводимости:

In [3]:
random_seed: int = 42

torch.manual_seed(random_seed)
np.random.seed(random_seed)
random.seed(random_seed)

if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

## Загрузка и подготовка данных CIFAR-10

In [4]:
class CIFAR10DataLoader:
    def __init__(self, batch_size: int, num_workers: int = 2, random_seed: int = 42) -> None:
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.random_seed = random_seed
        # ResNet18 uses this size as its input size.
        self.image_size = (224, 224)
        # For normalization.
        self.mean = (0.4914, 0.4822, 0.4465)
        self.std = (0.1953, 0.1925, 0.1942)

        # For reproducibility
        torch.manual_seed(self.random_seed)
        self.generator = torch.Generator().manual_seed(self.random_seed)

    def _get_transforms(self) -> Tuple[transforms.Compose, transforms.Compose]:
        """Creates transformations for training and test datasets."""
        transform_train = transforms.Compose([
            transforms.Resize(self.image_size),
            transforms.RandomCrop(self.image_size[0], padding=4),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(15),
            transforms.ToTensor(),
            transforms.Normalize(self.mean, self.std),
        ])

        transform_test = transforms.Compose([
            transforms.Resize(self.image_size),
            transforms.ToTensor(),
            transforms.Normalize(self.mean, self.std),
        ])

        return transform_train, transform_test

    def load_data(self) -> Tuple[DataLoader, DataLoader, DataLoader]:
        """Loads, splits, and preprocesses the CIFAR-10 dataset."""
        transform_train, transform_test = self._get_transforms()

        # Load full training set
        full_trainset = CIFAR10(root='./data', train=True,
                                download=True, transform=transform_train)
        testset = CIFAR10(root='./data', train=False,
                          download=True, transform=transform_test)

        # Define dataset sizes
        train_size = int(0.8 * len(full_trainset))
        val_size = len(full_trainset) - train_size

        # Randomly split into training and validation sets
        trainset, valset = random_split(
            full_trainset, [train_size, val_size], generator=self.generator)

        def seed_worker(worker_id: int) -> None:
            """Initializes seed for each worker in DataLoader."""
            worker_seed = torch.initial_seed() % 2**32
            np.random.seed(worker_seed)
            random.seed(worker_seed)

        g = torch.Generator()
        g.manual_seed(self.random_seed)

        # Create data loaders
        trainloader = DataLoader(trainset, batch_size=self.batch_size,
                                 shuffle=True, num_workers=self.num_workers, pin_memory=True,
                                 worker_init_fn=seed_worker, generator=g)
        valloader = DataLoader(valset, batch_size=self.batch_size,
                               shuffle=False, num_workers=self.num_workers, pin_memory=True,
                               worker_init_fn=seed_worker, generator=g)
        testloader = DataLoader(testset, batch_size=self.batch_size,
                                shuffle=False, num_workers=self.num_workers, pin_memory=True,
                                worker_init_fn=seed_worker, generator=g)

        return trainloader, valloader, testloader

## Определение архитектуры ResNet18 с SE-блоками

In [5]:
class SEBlock(nn.Module):
    def __init__(self, channel: int, reduction: int = 16) -> None:
        super(SEBlock, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)


class ResNet18SE(nn.Module):
    def __init__(
        self,
        device: torch.device = torch.device("cpu"),
        pretrained: bool = True,
        num_classes: int = 10
    ) -> None:
        """Initialize a ResNet18 model with attention."""
        super(ResNet18SE, self).__init__()
        weights = ResNet18_Weights.DEFAULT if pretrained else None
        self.resnet = models.resnet18(weights=weights)

        # Freeze weights
        for param in self.resnet.parameters():
            param.requires_grad = False

        # Add SE blocks after each convolutional block
        self._add_se_blocks()

        # Replace the final fully connected layer
        num_ftrs = self.resnet.fc.in_features
        self.resnet.fc = nn.Linear(num_ftrs, num_classes)

        # Move the model to the specified device
        self.resnet.to(device)
        if torch.__version__ >= "2.0":  # Use torch.compile() for optimization
            self.resnet = torch.compile(self.resnet)

    def _add_se_blocks(self) -> None:
        """Adds SE blocks after each convolutional block."""
        channels = [64, 128, 256, 512]
        for i, (layer, channel) in enumerate(zip(
                (self.resnet.layer1, self.resnet.layer2, self.resnet.layer3, self.resnet.layer4), channels)):
            layer.add_module(f"{i}_se", SEBlock(channel))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.resnet(x)

    def unfreeze_layers(self, num_layers: int) -> None:
        """Unfreezes the last num_layers layers for fine-tuning."""
        layers = [self.resnet.layer1, self.resnet.layer2,
                  self.resnet.layer3, self.resnet.layer4]
        for layer in layers[-num_layers:]:
            for param in layer.parameters():
                param.requires_grad = True

# Класс для обучения модели

In [6]:
class ModelTrainer:
    def __init__(self, model: nn.Module, device: torch.device, random_seed: int = 42):
        self.model = model
        self.device = device
        self.random_seed = random_seed
        torch.manual_seed(self.random_seed)

    def train(self, trainloader: DataLoader, valloader: DataLoader,
              learning_rate: float, num_epochs: int,
              weight_decay: float, patience: int) -> Tuple[List[float], List[float]]:
        """Trains the model and returns its training history."""
        self.model.to(self.device)
        criterion = nn.CrossEntropyLoss().to(self.device)
        optimizer = optim.Adam(self.model.parameters(),
                               lr=learning_rate, weight_decay=weight_decay)
        scheduler = ReduceLROnPlateau(
            optimizer, mode='max', factor=0.1, patience=patience // 2, min_lr=0)

        wandb.init(project="cifar10-optuna", config={
            "learning_rate": learning_rate,
            "epochs": num_epochs,
            "batch_size": trainloader.batch_size,
            "weight_decay": weight_decay
        })

        train_losses = []
        val_accuracies = []
        best_val_accuracy = float('-inf')
        epochs_no_improve = 0

        for epoch in range(num_epochs):
            self.model.train()
            running_loss = 0.0
            for inputs, labels in trainloader:
                inputs, labels = inputs.to(self.device), labels.to(self.device)

                optimizer.zero_grad()
                outputs = self.model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

                running_loss += loss.item()

            train_loss = running_loss / len(trainloader)
            train_losses.append(train_loss)
            print(
                f"Epoch {epoch+1}/{num_epochs}, Training Loss: {train_loss:.4f}")
            wandb.log({"train_loss": train_loss})

            # Evaluate on validation set
            val_accuracy = self.evaluate(valloader)
            val_accuracies.append(val_accuracy)
            wandb.log({"val_accuracy": val_accuracy})

            # Early stopping
            if val_accuracy > best_val_accuracy:
                best_val_accuracy = val_accuracy
                epochs_no_improve = 0
                best_model_state = self.model.state_dict()
            else:
                epochs_no_improve += 1
                if epochs_no_improve == patience:
                    print(
                        f"Early stopping! No improvement for {patience} epochs.")
                    self.model.load_state_dict(best_model_state)
                    break

            scheduler.step(val_accuracy)
            current_lr = scheduler.get_last_lr()[0]
            wandb.log({"lr": current_lr})
            if current_lr < learning_rate:
                print(f"Learning rate reduced to: {current_lr}")

        return train_losses, val_accuracies

    def evaluate(self, dataloader: DataLoader) -> float:
        """Evaluates the model on the given dataloader and returns the accuracy."""
        self.model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, labels in dataloader:
                inputs, labels = inputs.to(self.device), labels.to(self.device)
                outputs = self.model(inputs)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        accuracy = 100 * correct / total
        print(f"Validation Accuracy: {accuracy:.2f}%")
        return accuracy

# Оценка модели

In [7]:
def evaluate_test(
    model: torch.nn.Module,
    test_loader: DataLoader,
    device: torch.device,
) -> Tuple[List[int], List[int]]:
    """Evaluate a model on a given test dataset."""
    model.eval()
    true_labels = []
    predicted_labels = []
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            true_labels.extend(labels.cpu().numpy())
            predicted_labels.extend(predicted.cpu().numpy())

    return true_labels, predicted_labels

## Определение функции для оптимизации гиперпараметров

In [9]:
def objective(trial):
    """Optimize hyperparameters using Optuna."""
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    # Set hyperparameters
    hyperparams = {
        "learning_rate": trial.suggest_float("lr", 1e-5, 1e-4, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [32, 64]),
        "weight_decay": trial.suggest_float("wd", 1e-4, 5e-4, log=True),
        "num_epochs": trial.suggest_int("epochs", 12, 14),
        "patience": trial.suggest_int("patience", 5, 6),
        "num_layers_to_unfreeze": trial.suggest_int("num_layers_to_unfreeze", 2, 3)
    }
    num_workers = os.cpu_count() or 2

    # Load data
    loader = CIFAR10DataLoader(
        batch_size=hyperparams["batch_size"], num_workers=num_workers)
    train_loader, val_loader, _ = loader.load_data()

    # Initialize model
    model = ResNet18SE(device)

    # Freeze all layers
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze layers based on hyperparameter
    model.unfreeze_layers(num_layers=hyperparams["num_layers_to_unfreeze"])

    # Train model
    trainer = ModelTrainer(model, device)
    _, val_accuracy = trainer.train(
        train_loader, val_loader,
        learning_rate=hyperparams["learning_rate"],
        num_epochs=hyperparams["num_epochs"],
        weight_decay=hyperparams["weight_decay"],
        patience=hyperparams["patience"]
    )

    return max(val_accuracy)

## Запуск оптимизации

In [11]:
wandb.init()

# Create a study to maximize the metric
pruner = optuna.pruners.SuccessiveHalvingPruner()
study = optuna.create_study(direction="maximize", pruner=pruner)
study.optimize(objective, n_trials=5)

[I 2025-04-04 10:14:59,618] A new study created in memory with name: no-name-42a808b0-247d-4505-ac5e-faf2aec16d1e


Epoch 1/12, Training Loss: 0.6754
Validation Accuracy: 89.27%
Epoch 2/12, Training Loss: 0.3194
Validation Accuracy: 91.48%
Epoch 3/12, Training Loss: 0.2408
Validation Accuracy: 92.64%
Epoch 4/12, Training Loss: 0.1961
Validation Accuracy: 92.90%
Epoch 5/12, Training Loss: 0.1626
Validation Accuracy: 93.23%
Epoch 6/12, Training Loss: 0.1417
Validation Accuracy: 93.62%
Epoch 7/12, Training Loss: 0.1212
Validation Accuracy: 93.29%
Epoch 8/12, Training Loss: 0.1032
Validation Accuracy: 93.85%
Epoch 9/12, Training Loss: 0.0905
Validation Accuracy: 93.84%
Epoch 10/12, Training Loss: 0.0817
Validation Accuracy: 93.76%
Epoch 11/12, Training Loss: 0.0721
Validation Accuracy: 94.33%
Epoch 12/12, Training Loss: 0.0657


[I 2025-04-04 10:42:07,523] Trial 0 finished with value: 94.33 and parameters: {'lr': 3.60074082458154e-05, 'batch_size': 32, 'wd': 0.00021153179426531952, 'epochs': 12, 'patience': 5, 'num_layers_to_unfreeze': 2}. Best is trial 0 with value: 94.33.


Validation Accuracy: 94.06%
Epoch 1/12, Training Loss: 0.8702
Validation Accuracy: 88.53%
Epoch 2/12, Training Loss: 0.3683
Validation Accuracy: 91.59%
Epoch 3/12, Training Loss: 0.2668
Validation Accuracy: 93.03%
Epoch 4/12, Training Loss: 0.2157
Validation Accuracy: 93.28%
Epoch 5/12, Training Loss: 0.1794
Validation Accuracy: 93.95%
Epoch 6/12, Training Loss: 0.1550
Validation Accuracy: 94.36%
Epoch 7/12, Training Loss: 0.1331
Validation Accuracy: 94.46%
Epoch 8/12, Training Loss: 0.1144
Validation Accuracy: 94.80%
Epoch 9/12, Training Loss: 0.0988
Validation Accuracy: 94.53%
Epoch 10/12, Training Loss: 0.0868
Validation Accuracy: 94.86%
Epoch 11/12, Training Loss: 0.0761
Validation Accuracy: 94.92%
Epoch 12/12, Training Loss: 0.0659


[I 2025-04-04 11:14:54,425] Trial 1 finished with value: 95.09 and parameters: {'lr': 1.5334589134301916e-05, 'batch_size': 32, 'wd': 0.00033251891309273425, 'epochs': 12, 'patience': 5, 'num_layers_to_unfreeze': 3}. Best is trial 1 with value: 95.09.


...

Validation Accuracy: 96.40%
Learning rate reduced to: 6.5945635866259e-06
Epoch 1/12, Training Loss: 0.6064
Validation Accuracy: 89.77%
Epoch 2/12, Training Loss: 0.2926
Validation Accuracy: 91.75%
Epoch 3/12, Training Loss: 0.2211
Validation Accuracy: 92.80%
Epoch 4/12, Training Loss: 0.1779
Validation Accuracy: 92.87%
Epoch 5/12, Training Loss: 0.1478
Validation Accuracy: 93.37%
Epoch 6/12, Training Loss: 0.1285
Validation Accuracy: 93.57%
Epoch 7/12, Training Loss: 0.1094
Validation Accuracy: 93.53%
Epoch 8/12, Training Loss: 0.0932
Validation Accuracy: 93.75%
Epoch 9/12, Training Loss: 0.0841
Validation Accuracy: 93.52%
Epoch 10/12, Training Loss: 0.0753
Validation Accuracy: 93.78%
Epoch 11/12, Training Loss: 0.0657
Validation Accuracy: 94.07%
Epoch 12/12, Training Loss: 0.0617


[I 2025-04-04 12:19:26,532] Trial 3 finished with value: 94.18 and parameters: {'lr': 5.059971730926169e-05, 'batch_size': 32, 'wd': 0.00012578246687238484, 'epochs': 12, 'patience': 6, 'num_layers_to_unfreeze': 2}. Best is trial 2 with value: 96.4.


Validation Accuracy: 94.18%
Epoch 1/14, Training Loss: 0.6392
Validation Accuracy: 89.54%
Epoch 2/14, Training Loss: 0.3050
Validation Accuracy: 91.62%
Epoch 3/14, Training Loss: 0.2291
Validation Accuracy: 92.74%
Epoch 4/14, Training Loss: 0.1846
Validation Accuracy: 93.02%
Epoch 5/14, Training Loss: 0.1532
Validation Accuracy: 93.23%
Epoch 6/14, Training Loss: 0.1332
Validation Accuracy: 93.58%
Epoch 7/14, Training Loss: 0.1137
Validation Accuracy: 93.65%
Epoch 8/14, Training Loss: 0.0958
Validation Accuracy: 93.76%
Epoch 9/14, Training Loss: 0.0845
Validation Accuracy: 93.50%
Epoch 10/14, Training Loss: 0.0775
Validation Accuracy: 93.86%
Epoch 11/14, Training Loss: 0.0661
Validation Accuracy: 94.18%
Epoch 12/14, Training Loss: 0.0612
Validation Accuracy: 93.95%
Epoch 13/14, Training Loss: 0.0556
Validation Accuracy: 94.29%
Epoch 14/14, Training Loss: 0.0518


[I 2025-04-04 12:50:29,643] Trial 4 finished with value: 94.29 and parameters: {'lr': 4.246517590510316e-05, 'batch_size': 32, 'wd': 0.0001142012705766973, 'epochs': 14, 'patience': 6, 'num_layers_to_unfreeze': 2}. Best is trial 2 with value: 96.4.


Validation Accuracy: 94.21%


In [ ]:
best_params = study.best_params

with open("data/best_params.json", "w") as f:
    json.dump(best_params, f, indent=4)

print(f"Best params saved to best_params.json")

## Вывод результатов

In [15]:
print("Number of finished trials: {}".format(len(study.trials)))
print("Best trial:")
trial = study.best_trial
print("  Value: {}".format(trial.value))
print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

Number of finished trials: 5
Best trial:
  Value: 96.4
  Params: 
    lr: 6.5945635866259e-05
    batch_size: 32
    wd: 0.00011469106617371565
    epochs: 14
    patience: 5
    num_layers_to_unfreeze: 3


## Графики Accuracy и Loss по всем испытаниям из W&B

<figure>
  <img src="../data/val_accuracy.png?raw=true" loading="lazy" alt="val_accuracy" width="45%">
  <img src="../data/train_loss.png?raw=true" loading="lazy" alt="train_loss" width="45%">
</figure>